# Week 47, Day 4: Top-k problems

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w047d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import heapq

best = []                     # a min-heap of the 3 largest so far
for x in [4, 9, 1, 7, 3, 8, 2]:
    if len(best) < 3:
        heapq.heappush(best, x)
    elif x > best[0]:
        heapq.heapreplace(best, x)
    print(x, sorted(best), "weakest:", best[0])

In [ ]:
import heapq, random

def top_k(stream, k):
    heap = []
    for x in stream:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)
    return sorted(heap, reverse=True)

rng = random.Random(1)
stream = (rng.randint(0, 10**9) for _ in range(200_000))    # never stored as a list
print(top_k(stream, 5))

In [ ]:
import heapq
from collections import Counter

text = "the cat and the dog and the bird saw a cat and a dog"
counts = Counter(text.split())
print(counts.most_common(3))
print(heapq.nsmallest(3, counts.items(), key=lambda item: (-item[1], item[0])))

In [ ]:
import heapq

def nearest(points, target, k):
    tx, ty = target
    heap = []                                          # (-squared distance, index, point)
    for i, (x, y) in enumerate(points):
        d2 = (x - tx) ** 2 + (y - ty) ** 2
        if len(heap) < k:
            heapq.heappush(heap, (-d2, -i, (x, y)))
        elif -d2 > heap[0][0]:                         # closer than the farthest kept
            heapq.heapreplace(heap, (-d2, -i, (x, y)))
    return [p for _, _, p in sorted(heap, reverse=True)]

stations = [(5, 5), (1, 2), (-3, 4), (2, -1), (0, 6), (7, 0)]
print(nearest(stations, (0, 0), 3))

In [ ]:
import heapq

class RunningMedian:
    def __init__(self):
        self.low, self.high = [], []        # low stores negated values

    def add(self, x):
        heapq.heappush(self.low, -x)
        heapq.heappush(self.high, -heapq.heappop(self.low))
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):
            return -self.low[0]
        return (-self.low[0] + self.high[0]) / 2

m = RunningMedian()
for x in [5, 15, 1, 3, 8, 7, 9, 10]:
    m.add(x)
    print(x, m.median())

> **Key idea.** For the k largest, keep a min-heap of at most k items and replace its top whenever a newcomer beats it: O(n log k) time and O(k) memory, even on a stream. The k smallest or nearest is the mirror image (a max-heap, via negation, or `nsmallest`). For most frequent, count with `Counter`, then take the top k by `(-count, tie-breaker)`. Choose between sorting, a heap of size k, heapify-and-pop and quickselect by k, n and whether the data fits in memory. A running median keeps the lower half in a max-heap and the upper half in a min-heap, balanced to within one item.

### Exercise: Top k of a stream

Write `top_k(stream, k)`: the k largest items from any iterable, largest first, using a `heapq` min-heap that never holds more than k items (no `sorted` on the whole input, no `nlargest`). Then `bottom_k(stream, k)`: the k smallest, smallest first, using a max-heap of negated values. The checks feed you a generator of 100,000 numbers and watch the heap's size.

In [ ]:
import heapq

def top_k(stream, k):
    ___

def bottom_k(stream, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_topk():\n    \"streams\"\n    rng = random.Random(2)\n    data = [rng.randint(-10**6, 10**6) for _ in range(100_000)]\n    seen = [0]\n    def gen():\n        for x in data:\n            seen[0] += 1\n            yield x\n    assert top_k(gen(), 5) == sorted(data, reverse=True)[:5] and seen[0] == 100_000\n    assert bottom_k(iter(data), 4) == sorted(data)[:4]\n    assert top_k([3, 1], 5) == [3, 1] and top_k([], 3) == [] and bottom_k([2, 2, 1], 2) == [1, 2]\n    assert \"nlargest\" not in source and \"nsmallest\" not in source\ndef test_memory():\n    \"never more than k\"\n    biggest = [0]\n    import heapq as h\n    orig = h.heappush\n    def watched(heap, x):\n        orig(heap, x)\n        biggest[0] = max(biggest[0], len(heap))\n    h.heappush = watched\n    try:\n        top_k(range(10_000), 7)\n        bottom_k(range(10_000), 7)\n    finally:\n        h.heappush = orig\n    assert biggest[0] <= 7")

### Exercise: Most frequent

Write `most_frequent(words, k)`: the k most frequent words as `(word, count)` pairs, highest count first, ties in **alphabetical** order. Count with a `Counter`, then use `heapq.nsmallest` with a suitable key. Then `frequent_letters(text, k)`: the same for the letters in `text` (ignore anything that isn't a letter, and treat upper and lower case as the same).

In [ ]:
import heapq
from collections import Counter

def most_frequent(words, k):
    ___

def frequent_letters(text, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_frequent():\n    \"frequencies and ties\"\n    words = \"the cat and the dog and the bird saw a cat and a dog\".split()\n    assert most_frequent(words, 3) == [(\"and\", 3), (\"the\", 3), (\"a\", 2)]\n    assert most_frequent(words, 0) == [] and most_frequent([], 2) == []\n    assert frequent_letters(\"Hello, World!\", 3) == [(\"l\", 3), (\"o\", 2), (\"d\", 1)]")

### Exercise: The nearest few

Write `nearest(points, target, k)`: the k points nearest to `target` (2D tuples), nearest first, ties going to the point that appears **earlier** in the list. Keep a heap of at most k entries yourself (negated squared distances, with the index as a tie-breaker), and don't use square roots.

In [ ]:
import heapq

def nearest(points, target, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_nearest():\n    \"nearest\"\n    stations = [(5, 5), (1, 2), (-3, 4), (2, -1), (0, 6), (7, 0)]\n    assert nearest(stations, (0, 0), 3) == [(1, 2), (2, -1), (-3, 4)]\n    assert nearest([(1, 0), (0, 1), (-1, 0)], (0, 0), 2) == [(1, 0), (0, 1)]\n    assert nearest([], (0, 0), 3) == [] and nearest(stations, (0, 0), 0) == []\n    rng = random.Random(3)\n    for _ in range(100):\n        pts = [(rng.randint(-5, 5), rng.randint(-5, 5)) for _ in range(rng.randint(0, 20))]\n        t, k = (rng.randint(-5, 5), rng.randint(-5, 5)), rng.randint(0, 6)\n        d = lambda i: (pts[i][0] - t[0]) ** 2 + (pts[i][1] - t[1]) ** 2\n        want = [pts[i] for i in sorted(range(len(pts)), key=lambda i: (d(i), i))[:k]]\n        assert nearest(pts, t, k) == want\n    assert \"sqrt\" not in source and \"** 0.5\" not in source")

### Exercise: The kth largest so far

Write `KthLargest(k, initial=())`: `add(x)` adds a value and returns the kth largest value seen so far (or `None` while fewer than k values have been seen), each call O(log k), keeping a heap of at most k items. Also `kth()` returning the same without adding.

In [ ]:
import heapq

class KthLargest:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_kth():\n    \"kth largest\"\n    s = KthLargest(3, [4, 5, 8, 2])\n    assert s.kth() == 4\n    assert [s.add(x) for x in [3, 5, 10, 9, 4]] == [4, 5, 5, 8, 8]\n    t = KthLargest(2)\n    assert t.add(1) is None and t.add(5) == 1 and t.add(3) == 3 and t.kth() == 3\n    rng = random.Random(4)\n    for k in (1, 3, 10):\n        s, seen = KthLargest(k), []\n        for _ in range(500):\n            x = rng.randint(0, 1000)\n            seen.append(x)\n            want = sorted(seen, reverse=True)[k - 1] if len(seen) >= k else None\n            assert s.add(x) == want\n        heaps = [v for v in vars(s).values() if isinstance(v, list)]\n        assert heaps and len(heaps[0]) <= k")

### Exercise: The running median

Write `RunningMedian` with `add(x)`, `median()` (`ValueError` when empty; a whole value when the count is odd, the average of the two middle values otherwise), `len`, and `remove_median()`, which removes and returns the lower middle value (the median itself when the count is odd), keeping the halves balanced. Then `medians(xs)`: the list of medians after each item.

In [ ]:
import heapq

class RunningMedian:
    ___

def medians(xs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, statistics\ndef test_median():\n    \"medians\"\n    assert medians([5, 15, 1, 3, 8, 7, 9, 10]) == [5, 10.0, 5, 4.0, 5, 6.0, 7, 7.5]\n    m = RunningMedian()\n    try:\n        m.median()\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(5)\n    m, seen = RunningMedian(), []\n    for _ in range(2000):\n        if seen and rng.random() < 0.3:\n            seen.sort()\n            want = seen[(len(seen) - 1) // 2]\n            assert m.remove_median() == want\n            seen.remove(want)\n        else:\n            x = rng.randint(-100, 100)\n            m.add(x)\n            seen.append(x)\n        assert len(m) == len(seen)\n        if seen:\n            assert m.median() == statistics.median(seen)")

### Bug hunt: The wrong kind of heap

To keep the three largest, this keeps a heap whose top is the **largest** so far (by negating), and throws out the top when a better value arrives, so it throws out the best: this prints `[3, 2, 1]`. Keep a min-heap of the values themselves, so the weakest is on top, and it prints `[9, 8, 7]`.

In [ ]:
import heapq

best = []
for x in [4, 9, 1, 7, 3, 8, 2]:
    if len(best) < 3:
        heapq.heappush(best, -x)
    else:
        heapq.heappushpop(best, -x)
print(sorted((-v for v in best), reverse=True))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the weakest on top\"\n    expect_output(\"[9, 8, 7]\")")

### Bug hunt: Ties the wrong way round

This wants the most frequent words with ties in alphabetical order, but it negates nothing and uses `nlargest` on `(count, word)`, so among equal counts the alphabetically **last** word wins: this prints `['the', 'and', 'dog']`. Use `(-count, word)` with `nsmallest` so it prints `['and', 'the', 'a']`.

In [ ]:
import heapq
from collections import Counter

words = "the cat and the dog and the bird saw a cat and a dog".split()
counts = Counter(words)
top = heapq.nlargest(3, counts.items(), key=lambda item: (item[1], item[0]))
print([w for w, c in top])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"alphabetical ties\"\n    expect_output(\"['and', 'the', 'a']\")")

### Bug hunt: Halves that drift apart

This running median puts each value in whichever half it belongs to but never rebalances, so after a run of large values one heap holds nearly everything and its top isn't the middle: this prints `2`. Move items across so the halves differ by at most one, and it prints `4`.

In [ ]:
import heapq

low, high = [], []          # low: max-heap via negation; high: min-heap
for x in [1, 2, 3, 4, 5, 6, 7]:
    if not low or x <= -low[0]:
        heapq.heappush(low, -x)
    else:
        heapq.heappush(high, x)
print(-low[0] if len(low) >= len(high) else high[0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"balanced halves\"\n    expect_output(\"4\")")

### Puzzle: Keeping the k largest

Arrange the lines into a top-k function. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# A min-heap of size 2 keeps the 2 largest of the stream 6, 2, 8, 5, 9, 1.

p = ___   # the heap's top after the first three values (6, 2, 8)
q = ___   # the heap's top at the end
r = ___   # the median of [5, 15, 1, 3]
s = ___   # how many items the low half holds after 7 values (low may have one extra)
t = ___   # the memory a size-k heap needs for top k of n items, like "O(k)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 6 and q == 8 and r == 4.0, \"p, q and r\"\n    assert s == 4 and t.replace(\" \", \"\") == \"O(k)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the smallest sums

Given two sorted lists `a` and `b`, find the k pairs `(x, y)` (x from `a`, y from `b`) with the smallest sums, smallest first; ties go to the smaller index in `a`, then in `b`. There are len(a) × len(b) pairs, far too many to build when the lists are long.

Think of the pairs as a grid: row i is `a[i]` paired with each `b[j]`, and each row is sorted. So the smallest pair overall is (0, 0), and the next smallest is always the **next unused pair in some row**. Keep a heap holding, for each row, its next candidate: start with `(a[i] + b[0], i, 0)` for the first `min(k, len(a))` rows; pop the smallest, record it, and push the next pair in the same row, `(i, j + 1)`. That's O(k log k): the heap never holds more than k entries.

Write `smallest_pairs(a, b, k)` returning a list of `(x, y)` pairs (fewer than k if there aren't enough), and `kth_smallest_sum(a, b, k)` (`IndexError` if k is too big). The checks use lists of 100,000 items each with k = 1,000.

In [ ]:
import heapq

def smallest_pairs(a, b, k):
    ___

def kth_smallest_sum(a, b, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_small():\n    \"examples\"\n    assert smallest_pairs([1, 7, 11], [2, 4, 6], 3) == [(1, 2), (1, 4), (1, 6)]\n    assert smallest_pairs([1, 1, 2], [1, 2, 3], 2) == [(1, 1), (1, 1)]\n    assert smallest_pairs([1, 2], [3], 3) == [(1, 3), (2, 3)] and smallest_pairs([], [1], 2) == []\n    assert kth_smallest_sum([1, 7, 11], [2, 4, 6], 4) == 9\n    try:\n        kth_smallest_sum([1], [2], 2)\n        assert False\n    except IndexError:\n        pass\ndef test_random():\n    \"against brute force\"\n    rng = random.Random(6)\n    for _ in range(200):\n        a = sorted(rng.randint(-20, 20) for _ in range(rng.randint(0, 8)))\n        b = sorted(rng.randint(-20, 20) for _ in range(rng.randint(0, 8)))\n        k = rng.randint(0, 70)\n        allp = sorted((a[i] + b[j], i, j) for i in range(len(a)) for j in range(len(b)))\n        assert smallest_pairs(a, b, k) == [(a[i], b[j]) for _, i, j in allp[:k]]\ndef test_big():\n    \"big lists\"\n    a = list(range(0, 300_000, 3))\n    b = list(range(0, 200_000, 2))\n    t0 = time.perf_counter()\n    got = smallest_pairs(a, b, 1000)\n    assert len(got) == 1000 and got[0] == (0, 0) and time.perf_counter() - t0 < 2\n    sums = [x + y for x, y in got]\n    assert sums == sorted(sums)")

## Warm-up

At every moment the heap holds the three largest values seen so far, and its top, `best[0]`, is the **weakest** of them: the one a newcomer has to beat. Seven values went past, but the heap never held more than three.

## The k largest of a stream

To find the k largest of n items:

- keep a **min-heap** of at most k items;
- for each new item: if the heap has fewer than k, push it; otherwise, if it beats the weakest (`x > heap[0]`), replace the weakest with it.

Each step is O(log k), so the whole job is **O(n log k)** time and **O(k)** memory. When the items come from a file or a network, that memory bound is what matters: you can find the hundred largest of a billion numbers with a heap of a hundred.

Why a min-heap for the largest? Because the question at each step is "is this new item good enough to get in?", and the answer depends only on the weakest member, which a min-heap keeps on top. A max-heap would put the best on top, which you don't need until the end.

At the end, the heap isn't sorted: pop it (smallest first) and reverse, or call `sorted(heap, reverse=True)`, an O(k log k) step that's small when k is.

That's what `heapq.nlargest(k, iterable)` does (with extra care to keep ties in their original order). `nsmallest` is the mirror image, with a max-heap of size k.

## Most frequent

For "the k most frequent words", count first with a `Counter` (Week 20), then take the top k of the counts. `Counter.most_common(k)` does exactly that (with `heapq.nlargest` inside). Ties need a rule, and `most_common` breaks them by the order the items were **first counted**; often you want something you can state, such as "alphabetical among equal counts". Then the heap's key needs two parts that point in **different directions** (higher count better, earlier word better), which a tuple `(-count, word)` gives you with `nsmallest`:

## Nearest

"The k nearest" is the k **smallest** distances. With `heapq.nsmallest(k, points, key=distance)` it's one line. When writing the heap yourself, the usual trick is to keep a **max**-heap of size k by pushing **negated** distances, so the farthest of your current k is on top, ready to be replaced. Distances only need comparing, not computing exactly, so skip the square root: if a² < b² then a < b for distances.

The index `-i` inside the tuple breaks ties between equal distances in favour of the **earlier** point (in a max-heap of negatives, a larger `-i` means a smaller `i`).

## Choosing a method

For the k largest of n items in a list:

| method | time | memory | when |
|---|---|---|---|
| `sorted(xs)[-k:]` | O(n log n) | O(n) | k close to n, or you need it all sorted anyway |
| heap of size k (`nlargest`) | O(n log k) | O(k) | k much smaller than n, or a stream |
| heapify + k pops (Day 3) | O(n + k log n) | O(n) | the whole list is in memory |
| quickselect (Week 19 Day 2) | O(n) on average | O(n) | you only need the kth value, or the top k unsorted |
| `max(xs)` | O(n) | O(1) | k = 1 |

Only the heap of size k works on a stream you can't store.

## The running median

The **median** of a collection is its middle value (or the average of the two middle values when there's an even number). To keep it up to date as numbers arrive, split the numbers seen so far into two halves:

- a **max-heap** `low` holding the smaller half (its top is the largest of the small ones);
- a **min-heap** `high` holding the larger half (its top is the smallest of the large ones).

Keep two rules: every value in `low` is ≤ every value in `high`, and the sizes differ by at most one (say `low` may have one extra). Then the median is `low`'s top, or the average of both tops. Adding a number: push it into `low`, move `low`'s largest across to `high` (which keeps the order rule), and if `high` is now bigger, move its smallest back. Each step is O(log n).

```text
      low (max-heap)          high (min-heap)
   [ 1  3  4  (5) ]  |  [ (7)  8  9 ]        median 5
            top ^           ^ top
```

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a leaderboard you've seen (a game, a fitness app). How would you keep its top ten up to date as scores arrive, and what would you do about a player whose score changes? Tomorrow: merging k sorted lists with a heap, the job `heapq.merge` does.